In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"


spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}"
)


print(
    f"Bronze source: {catalog}.{bronze_schema}"
)

print(
    f"Silver destination: {catalog}.{silver_schema}"
)


display(
    spark.sql(
        f"SHOW TABLES IN {catalog}.{bronze_schema}"
    )
)

Bronze source: crypto_lakehouse.bronze
Silver destination: crypto_lakehouse.silver


database,tableName,isTemporary
bronze,binance_account_snapshots,false
bronze,binance_convert_trades,false
bronze,binance_deposits,false
bronze,binance_funding_account_snapshots,false
bronze,binance_p2p_trades,false
bronze,binance_price_history,false
bronze,binance_spot_trades,false
bronze,binance_transfers,false
bronze,binance_withdrawals,false
bronze,coindcx_balances,false


#### Binance trades

In [0]:
spot_bronze_df = spark.table(f"{catalog}.{bronze_schema}.binance_spot_trades")
convert_bronze_df = spark.table(f"{catalog}.{bronze_schema}.binance_convert_trades")

print("Spot rows",spot_bronze_df.count())
print("Convert rows",convert_bronze_df.count())

display(spot_bronze_df.limit(5))
display(convert_bronze_df.limit(5))

Spot rows 13
Convert rows 3


symbol,id,orderId,orderListId,price,qty,quoteQty,commission,commissionAsset,time,isBuyer,isMaker,isBestMatch,source_file_name,_source_file,exchange,source_type,ingested_at
BTCUSDT,6636792207,66035545193,-1,79121.50000000,0.00007000,5.53850500,0.00000007,BTC,1788114758003,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:33.930Z
BTCUSDT,6497931425,64430221057,-1,64266.00000000,0.00010000,6.42660000,0.00000010,BTC,1783798810981,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:33.930Z
BTCUSDT,6369541361,62793879136,-1,60820.00000000,0.00008000,4.86560000,0.00000008,BTC,1780668810940,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:33.930Z
BTCUSDT,6636792208,66035545193,-1,79121.50000000,0.00004000,3.16486000,0.00000004,BTC,1788114758003,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:33.930Z
SOLUSDT,1972197390,17225421892,-1,64.92000000,0.07900000,5.12868000,0.00007900,SOL,1780668893697,true,false,true,trades_20260925T182106_413092Z_d4c5c822.json,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,Binance,Spot_trade,2026-10-02T14:21:33.930Z


quoteId,orderId,orderStatus,fromAsset,fromAmount,toAsset,toAmount,ratio,inverseRatio,createTime,orderType,side,walletType,source_file_name,_source_file,exchange,source_type,ingested_at
T69B1DEEB_B000000000006A953,2221355118396379831,SUCCESS,USDT,1.95,SOL,0.02244853,0.0115121,86.8654,1773335771268,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:46.415Z
T69B3306C_B0000000000038786,2221766670752487619,SUCCESS,USDT,3.54,SOL,0.03993758,0.0112818,88.6383,1773383682256,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:46.415Z
T69B3306C_B0000000000036979,2221756019233545619,SUCCESS,USDT,4.04000058,SOL,0.04526871,0.0112051,89.2449,1773382442380,MARKET,BUY,SPOT,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:46.415Z


In [0]:
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


decimal_type = DecimalType(38, 18)


spot_clean_df = (
    spot_bronze_df
    .select(
        col("id").cast("string").alias("transaction_id"),
        col("orderId").cast("string").alias("order_id"),
        col("symbol"),
        when(
            col("isBuyer"),
            lit("BUY")
        ).otherwise(
            lit("SELL")
        ).alias("side"),
        col("price").cast(decimal_type).alias("price"),
        col("qty").cast(decimal_type).alias("quantity"),
        col("quoteQty").cast(decimal_type).alias("quote_amount"),
        col("commission").cast(decimal_type).alias("fee_amount"),
        col("commissionAsset").alias("fee_asset"),
        col("isMaker").alias("is_maker"),
        timestamp_millis(
            col("time")
        ).alias("trade_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)


spot_clean_df.printSchema()

display(
    spot_clean_df.orderBy(
        col("trade_time").desc()
    )
)

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- side: string (nullable = false)
 |-- price: decimal(38,18) (nullable = true)
 |-- quantity: decimal(38,18) (nullable = true)
 |-- quote_amount: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,18) (nullable = true)
 |-- fee_asset: string (nullable = true)
 |-- is_maker: boolean (nullable = true)
 |-- trade_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



transaction_id,order_id,symbol,side,price,quantity,quote_amount,fee_amount,fee_asset,is_maker,trade_time,exchange,_source_file,bronze_ingested_at,silver_processed_at
6636792207,66035545193,BTCUSDT,BUY,79121.500000000000000000,0.000070000000000000,5.538505000000000000,7.0000000000E-8,BTC,false,2026-08-30T18:32:38.003Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
6636792208,66035545193,BTCUSDT,BUY,79121.500000000000000000,0.000040000000000000,3.164860000000000000,4.0000000000E-8,BTC,false,2026-08-30T18:32:38.003Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
4255932876,48872688120,ETHUSDT,BUY,1878.410000000000000000,0.004000000000000000,7.513640000000000000,0.000004000000000000,ETH,false,2026-08-05T15:26:48.425Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
6497931425,64430221057,BTCUSDT,BUY,64266.000000000000000000,0.000100000000000000,6.426600000000000000,1.00000000000E-7,BTC,false,2026-07-11T19:40:10.981Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
60681896,994044647,ONDOUSDT,BUY,0.337100000000000000,18.300000000000000000,6.168930000000000000,0.018300000000000000,ONDO,false,2026-07-04T22:24:18.443Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ONDOUSDT/trades_20260925T182107_841454Z_e13b5606.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
472919659,7811815957,LINKUSDT,BUY,7.919000000000000000,0.640000000000000000,5.068160000000000000,0.000640000000000000,LINK,false,2026-06-14T09:38:33.928Z,Binance,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
472919660,7811815957,LINKUSDT,BUY,7.919000000000000000,0.120000000000000000,0.950280000000000000,0.000120000000000000,LINK,false,2026-06-14T09:38:33.928Z,Binance,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
523831127,8625895369,SUIUSDT,BUY,0.749600000000000000,8.100000000000000000,6.071760000000000000,0.008100000000000000,SUI,false,2026-06-11T15:50:18.493Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SUIUSDT/trades_20260925T182106_890100Z_aa258a00.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
4121758522,47509568709,ETHUSDT,BUY,1658.290000000000000000,0.003000000000000000,4.974870000000000000,0.000003000000000000,ETH,false,2026-06-11T06:28:07.152Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z
1972197389,17225421892,SOLUSDT,BUY,64.920000000000000000,0.027000000000000000,1.752840000000000000,0.000027000000000000,SOL,false,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,2026-10-02T14:44:41.932Z


In [0]:
# Normalize Spot purchases and sales

spot_normalized_df = (
    spot_clean_df
    .withColumn(
        "base_asset",
        regexp_replace(
            col("symbol"),
            "USDT$",
            ""
        )
    )
    .withColumn(
        "quote_asset",
        lit("USDT")
    )
    .withColumn(
        "asset_spent",
        when(
            col("side") == "BUY",
            col("quote_asset")
        ).otherwise(
            col("base_asset")
        )
    )
    .withColumn(
        "amount_spent",
        when(
            col("side") == "BUY",
            col("quote_amount")
        ).otherwise(
            col("quantity")
        )
    )
    .withColumn(
        "asset_received",
        when(
            col("side") == "BUY",
            col("base_asset")
        ).otherwise(
            col("quote_asset")
        )
    )
    .withColumn(
        "amount_received",
        when(
            col("side") == "BUY",
            col("quantity")
        ).otherwise(
            col("quote_amount")
        )
    )
    .withColumn(
        "trade_source",
        lit("Spot")
    )
)


display(
    spot_normalized_df.select(
        "transaction_id",
        "side",
        "asset_spent",
        "amount_spent",
        "asset_received",
        "amount_received",
        "price",
        "fee_amount",
        "fee_asset",
        "trade_time"
    )
)

transaction_id,side,asset_spent,amount_spent,asset_received,amount_received,price,fee_amount,fee_asset,trade_time
6636792207,BUY,USDT,5.538505000000000000,BTC,0.000070000000000000,79121.500000000000000000,7.0000000000E-8,BTC,2026-08-30T18:32:38.003Z
6497931425,BUY,USDT,6.426600000000000000,BTC,0.000100000000000000,64266.000000000000000000,1.00000000000E-7,BTC,2026-07-11T19:40:10.981Z
6369541361,BUY,USDT,4.865600000000000000,BTC,0.000080000000000000,60820.000000000000000000,8.0000000000E-8,BTC,2026-06-05T14:13:30.940Z
6636792208,BUY,USDT,3.164860000000000000,BTC,0.000040000000000000,79121.500000000000000000,4.0000000000E-8,BTC,2026-08-30T18:32:38.003Z
1972197390,BUY,USDT,5.128680000000000000,SOL,0.079000000000000000,64.920000000000000000,0.000079000000000000,SOL,2026-06-05T14:14:53.697Z
1972197389,BUY,USDT,1.752840000000000000,SOL,0.027000000000000000,64.920000000000000000,0.000027000000000000,SOL,2026-06-05T14:14:53.697Z
1972197391,BUY,USDT,0.584280000000000000,SOL,0.009000000000000000,64.920000000000000000,0.000009000000000000,SOL,2026-06-05T14:14:53.697Z
4121758522,BUY,USDT,4.974870000000000000,ETH,0.003000000000000000,1658.290000000000000000,0.000003000000000000,ETH,2026-06-11T06:28:07.152Z
4255932876,BUY,USDT,7.513640000000000000,ETH,0.004000000000000000,1878.410000000000000000,0.000004000000000000,ETH,2026-08-05T15:26:48.425Z
472919659,BUY,USDT,5.068160000000000000,LINK,0.640000000000000000,7.919000000000000000,0.000640000000000000,LINK,2026-06-14T09:38:33.928Z


In [0]:
convert_clean_df = (
    convert_bronze_df
    .select(
        col("orderId")
            .cast("string")
            .alias("transaction_id"),

        lit("Convert")
            .alias("trade_source"),

        lit(None)
            .cast("string")
            .alias("symbol"),

        col("side"),

        col("fromAsset")
            .alias("asset_spent"),

        col("fromAmount")
            .cast(decimal_type)
            .alias("amount_spent"),

        col("toAsset")
            .alias("asset_received"),

        col("toAmount")
            .cast(decimal_type)
            .alias("amount_received"),

        lit(None)
            .cast(decimal_type)
            .alias("fee_amount"),

        lit(None)
            .cast("string")
            .alias("fee_asset"),

        timestamp_millis(
            col("createTime")
        ).alias("trade_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
            .alias("bronze_ingested_at")
    )
)

display(convert_clean_df)

transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at
2221355118396379831,Convert,null,BUY,USDT,1.950000000000000000,SOL,0.022448530000000000,null,null,2026-03-12T17:16:11.268Z,Binance,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,2026-10-02T14:21:46.415Z
2221766670752487619,Convert,null,BUY,USDT,3.540000000000000000,SOL,0.039937580000000000,null,null,2026-03-13T06:34:42.256Z,Binance,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,2026-10-02T14:21:46.415Z
2221756019233545619,Convert,null,BUY,USDT,4.040000580000000000,SOL,0.045268710000000000,null,null,2026-03-13T06:14:02.380Z,Binance,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,2026-10-02T14:21:46.415Z


In [0]:
spot_for_trades_df = (
    spot_normalized_df
    .select(
        col("transaction_id"),
        col("trade_source"),
        col("symbol"),
        col("side"),

        col("asset_spent"),
        col("amount_spent"),

        col("asset_received"),
        col("amount_received"),

        col("fee_amount"),
        col("fee_asset"),

        col("trade_time"),
        col("exchange"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
)

display(spot_for_trades_df)

transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at
6636792207,Spot,BTCUSDT,BUY,USDT,5.538505000000000000,BTC,0.000070000000000000,7.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z
6497931425,Spot,BTCUSDT,BUY,USDT,6.426600000000000000,BTC,0.000100000000000000,1.00000000000E-7,BTC,2026-07-11T19:40:10.981Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z
6369541361,Spot,BTCUSDT,BUY,USDT,4.865600000000000000,BTC,0.000080000000000000,8.0000000000E-8,BTC,2026-06-05T14:13:30.940Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z
6636792208,Spot,BTCUSDT,BUY,USDT,3.164860000000000000,BTC,0.000040000000000000,4.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z
1972197390,Spot,SOLUSDT,BUY,USDT,5.128680000000000000,SOL,0.079000000000000000,0.000079000000000000,SOL,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z
1972197389,Spot,SOLUSDT,BUY,USDT,1.752840000000000000,SOL,0.027000000000000000,0.000027000000000000,SOL,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z
1972197391,Spot,SOLUSDT,BUY,USDT,0.584280000000000000,SOL,0.009000000000000000,0.000009000000000000,SOL,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z
4121758522,Spot,ETHUSDT,BUY,USDT,4.974870000000000000,ETH,0.003000000000000000,0.000003000000000000,ETH,2026-06-11T06:28:07.152Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z
4255932876,Spot,ETHUSDT,BUY,USDT,7.513640000000000000,ETH,0.004000000000000000,0.000004000000000000,ETH,2026-08-05T15:26:48.425Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z
472919659,Spot,LINKUSDT,BUY,USDT,5.068160000000000000,LINK,0.640000000000000000,0.000640000000000000,LINK,2026-06-14T09:38:33.928Z,Binance,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z


In [0]:
silver_trades_df = (
    spot_for_trades_df
    .unionByName(
        convert_clean_df
    )
    .withColumn(
        "trade_key",
        concat_ws(
            "|",
            col("exchange"),
            col("trade_source"),
            coalesce(
                col("symbol"),
                lit("NO_SYMBOL")
            ),
            col("transaction_id")
        )
    )
)

print("Total Silver trades:", silver_trades_df.count())

display(
    silver_trades_df
    .select(
        "trade_key",
        "transaction_id",
        "trade_source",
        "symbol",
        "side",
        "asset_spent",
        "amount_spent",
        "asset_received",
        "amount_received",
        "trade_time"
    )
    .orderBy(
        col("trade_time").desc()
    )
)

Total Silver trades: 16


trade_key,transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,trade_time
Binance|Spot|BTCUSDT|6636792207,6636792207,Spot,BTCUSDT,BUY,USDT,5.538505000000000000,BTC,0.000070000000000000,2026-08-30T18:32:38.003Z
Binance|Spot|BTCUSDT|6636792208,6636792208,Spot,BTCUSDT,BUY,USDT,3.164860000000000000,BTC,0.000040000000000000,2026-08-30T18:32:38.003Z
Binance|Spot|ETHUSDT|4255932876,4255932876,Spot,ETHUSDT,BUY,USDT,7.513640000000000000,ETH,0.004000000000000000,2026-08-05T15:26:48.425Z
Binance|Spot|BTCUSDT|6497931425,6497931425,Spot,BTCUSDT,BUY,USDT,6.426600000000000000,BTC,0.000100000000000000,2026-07-11T19:40:10.981Z
Binance|Spot|ONDOUSDT|60681896,60681896,Spot,ONDOUSDT,BUY,USDT,6.168930000000000000,ONDO,18.300000000000000000,2026-07-04T22:24:18.443Z
Binance|Spot|LINKUSDT|472919659,472919659,Spot,LINKUSDT,BUY,USDT,5.068160000000000000,LINK,0.640000000000000000,2026-06-14T09:38:33.928Z
Binance|Spot|LINKUSDT|472919660,472919660,Spot,LINKUSDT,BUY,USDT,0.950280000000000000,LINK,0.120000000000000000,2026-06-14T09:38:33.928Z
Binance|Spot|SUIUSDT|523831127,523831127,Spot,SUIUSDT,BUY,USDT,6.071760000000000000,SUI,8.100000000000000000,2026-06-11T15:50:18.493Z
Binance|Spot|ETHUSDT|4121758522,4121758522,Spot,ETHUSDT,BUY,USDT,4.974870000000000000,ETH,0.003000000000000000,2026-06-11T06:28:07.152Z
Binance|Spot|SOLUSDT|1972197391,1972197391,Spot,SOLUSDT,BUY,USDT,0.584280000000000000,SOL,0.009000000000000000,2026-06-05T14:14:53.697Z


In [0]:
trades_before = silver_trades_df.count()

silver_trades_clean_df = (
    silver_trades_df
    .filter(
        col("transaction_id").isNotNull()
        & col("asset_spent").isNotNull()
        & col("asset_received").isNotNull()
        & (col("amount_spent") > 0)
        & (col("amount_received") > 0)
    )
    .dropDuplicates([
        "trade_key"
    ])
    .withColumn(
        "trade_date",
        to_date(col("trade_time"))
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)


trades_after = silver_trades_clean_df.count()


print("Rows before cleaning:", trades_before)
print("Rows after cleaning:", trades_after)
print("Rows removed:", trades_before - trades_after)

Rows before cleaning: 16
Rows after cleaning: 16
Rows removed: 0


In [0]:
(
    silver_trades_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_trades"
    )
)

In [0]:
display(
    spark.table(
        f"{catalog}.{silver_schema}.binance_trades"
    )
)

transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at,trade_key,trade_date,silver_processed_at
6636792207,Spot,BTCUSDT,BUY,USDT,5.538505000000000000,BTC,0.000070000000000000,7.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,Binance|Spot|BTCUSDT|6636792207,2026-08-30,2026-10-02T14:44:52.520Z
1972197390,Spot,SOLUSDT,BUY,USDT,5.128680000000000000,SOL,0.079000000000000000,0.000079000000000000,SOL,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,Binance|Spot|SOLUSDT|1972197390,2026-06-05,2026-10-02T14:44:52.520Z
6636792208,Spot,BTCUSDT,BUY,USDT,3.164860000000000000,BTC,0.000040000000000000,4.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,Binance,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,Binance|Spot|BTCUSDT|6636792208,2026-08-30,2026-10-02T14:44:52.520Z
4255932876,Spot,ETHUSDT,BUY,USDT,7.513640000000000000,ETH,0.004000000000000000,0.000004000000000000,ETH,2026-08-05T15:26:48.425Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,Binance|Spot|ETHUSDT|4255932876,2026-08-05,2026-10-02T14:44:52.520Z
472919660,Spot,LINKUSDT,BUY,USDT,0.950280000000000000,LINK,0.120000000000000000,0.000120000000000000,LINK,2026-06-14T09:38:33.928Z,Binance,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,Binance|Spot|LINKUSDT|472919660,2026-06-14,2026-10-02T14:44:52.520Z
4121758522,Spot,ETHUSDT,BUY,USDT,4.974870000000000000,ETH,0.003000000000000000,0.000003000000000000,ETH,2026-06-11T06:28:07.152Z,Binance,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,Binance|Spot|ETHUSDT|4121758522,2026-06-11,2026-10-02T14:44:52.520Z
472919659,Spot,LINKUSDT,BUY,USDT,5.068160000000000000,LINK,0.640000000000000000,0.000640000000000000,LINK,2026-06-14T09:38:33.928Z,Binance,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,Binance|Spot|LINKUSDT|472919659,2026-06-14,2026-10-02T14:44:52.520Z
523831127,Spot,SUIUSDT,BUY,USDT,6.071760000000000000,SUI,8.100000000000000000,0.008100000000000000,SUI,2026-06-11T15:50:18.493Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SUIUSDT/trades_20260925T182106_890100Z_aa258a00.json,2026-10-02T14:21:33.930Z,Binance|Spot|SUIUSDT|523831127,2026-06-11,2026-10-02T14:44:52.520Z
1972197389,Spot,SOLUSDT,BUY,USDT,1.752840000000000000,SOL,0.027000000000000000,0.000027000000000000,SOL,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,Binance|Spot|SOLUSDT|1972197389,2026-06-05,2026-10-02T14:44:52.520Z
1972197391,Spot,SOLUSDT,BUY,USDT,0.584280000000000000,SOL,0.009000000000000000,0.000009000000000000,SOL,2026-06-05T14:14:53.697Z,Binance,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,Binance|Spot|SOLUSDT|1972197391,2026-06-05,2026-10-02T14:44:52.520Z


####binance p2p

In [0]:
p2p_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_p2p_trades"
)


print(
    "P2P Bronze rows:",
    p2p_bronze_df.count()
)


display(
    p2p_bronze_df
)

P2P Bronze rows: 15


orderNumber,advNo,tradeType,asset,fiat,fiatSymbol,amount,totalPrice,unitPrice,orderStatus,createTime,takerCommissionRate,takerCommission,takerAmount,counterPartNickName,payMethodName,additionalKycVerify,source_file_name,_source_file,exchange,source_type,ingested_at
22899404533812277248,12892779563683180544,BUY,USDT,INR,₹,6.08000000,630.00000000,103.54,COMPLETED,1781428755226,0,0.06,6.02,Tra***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22898272405559799808,12892779563683180544,BUY,USDT,INR,₹,5.07000000,530.00000000,104.45,COMPLETED,1781158834842,0,0.06,5.01,Tra***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22896216150807388160,12889979499159543808,BUY,USDT,INR,₹,2.89000000,300.00000000,103.7,COMPLETED,1780668585470,0,0.06,2.83,Tra***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22909209175754317824,12908121022744678400,BUY,USDT,INR,₹,3.42000000,350.00000000,102.2,COMPLETED,1783766364047,0,0.06,3.36,No-***,UPIQRCode,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22906848402065932288,12904794820655955968,BUY,USDT,INR,₹,3.39000000,350.00000000,103,COMPLETED,1783203511749,0,0.06,3.33,No-***,UPIQRCode,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22918338709083369472,12828747426765901824,BUY,USDT,INR,₹,3.88000000,400.00000000,102.94,COMPLETED,1785943014361,0,0.06,3.82,DL8***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22898377650986762240,12898377207256502272,BUY,USDT,INR,₹,3.84000000,400.00000000,104,CANCELLED,1781183927316,0,0.06,3.78,pan***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22924149347226902528,12908479299326263296,BUY,USDT,INR,₹,3.86000000,400.00000000,103.5,COMPLETED,1787328378475,0,0.06,3.8,P2P***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22895954482175311872,12893354595859169280,BUY,USDT,INR,₹,4.80000000,500.00000000,104,CANCELLED,1780606198817,0,0.06,4.74,-Dk***,UPIQRCode,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z
22907162745527037952,12905900830268932096,BUY,USDT,INR,₹,3.50000000,359.00000000,102.3,COMPLETED,1783278457023,0,0.06,3.44,TRU***,UPIQRCode,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:07.631Z


In [0]:
p2p_silver_df = (
    p2p_bronze_df
    .select(
        col("orderNumber")
        .alias("p2p_order_id"),

        col("tradeType")
        .alias("trade_type"),

        col("asset"),

        col("fiat"),

        col("amount")
        .cast(decimal_type)
        .alias("asset_amount"),

        col("totalPrice")
        .cast(decimal_type)
        .alias("fiat_amount"),

        col("unitPrice")
        .cast(decimal_type)
        .alias("unit_price"),

        col("takerCommission")
        .cast(decimal_type)
        .alias("fee_amount"),

        col("takerAmount")
        .cast(decimal_type)
        .alias("net_asset_amount"),

        col("orderStatus")
        .alias("status"),

        timestamp_millis(
            col("createTime")
        ).alias("transaction_time"),

        col("payMethodName")
        .alias("payment_method"),

        col("exchange"),

        col("_source_file"),

        col("ingested_at").alias(
            "bronze_ingested_at"
        )
    )
    .dropDuplicates([
        "p2p_order_id"
    ])
    .withColumn(
        "is_completed",
        col("status") == "COMPLETED"
    )
    .withColumn(
        "transaction_date",
        to_date(col("transaction_time"))
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)

print(
    "P2P Silver rows:",
    p2p_silver_df.count()
)

display(
    p2p_silver_df.orderBy(
        col("transaction_time").desc()
    )
)

P2P Silver rows: 15


p2p_order_id,trade_type,asset,fiat,asset_amount,fiat_amount,unit_price,fee_amount,net_asset_amount,status,transaction_time,payment_method,exchange,_source_file,bronze_ingested_at,is_completed,transaction_date,silver_processed_at
22927112581471313920,BUY,USDT,INR,5.330000000000000000,550.000000000000000000,103.000000000000000000,0.060000000000000000,5.270000000000000000,COMPLETED,2026-08-29T20:21:08.551Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-08-29,2026-10-02T14:44:58.547Z
22924149347226902528,BUY,USDT,INR,3.860000000000000000,400.000000000000000000,103.500000000000000000,0.060000000000000000,3.800000000000000000,COMPLETED,2026-08-21T16:06:18.475Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-08-21,2026-10-02T14:44:58.547Z
22918338709083369472,BUY,USDT,INR,3.880000000000000000,400.000000000000000000,102.940000000000000000,0.060000000000000000,3.820000000000000000,COMPLETED,2026-08-05T15:16:54.361Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-08-05,2026-10-02T14:44:58.547Z
22910348341135761408,BUY,USDT,INR,3.420000000000000000,350.000000000000000000,102.300000000000000000,0.060000000000000000,3.360000000000000000,COMPLETED,2026-07-14T14:06:02.235Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-14,2026-10-02T14:44:58.547Z
22909209175754317824,BUY,USDT,INR,3.420000000000000000,350.000000000000000000,102.200000000000000000,0.060000000000000000,3.360000000000000000,COMPLETED,2026-07-11T10:39:24.047Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-11,2026-10-02T14:44:58.547Z
22907162745527037952,BUY,USDT,INR,3.500000000000000000,359.000000000000000000,102.300000000000000000,0.060000000000000000,3.440000000000000000,COMPLETED,2026-07-05T19:07:37.023Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-05,2026-10-02T14:44:58.547Z
22906848752644837376,BUY,USDT,INR,2.910000000000000000,300.000000000000000000,102.990000000000000000,0.060000000000000000,2.850000000000000000,COMPLETED,2026-07-04T22:19:55.318Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-04,2026-10-02T14:44:58.547Z
22906848402065932288,BUY,USDT,INR,3.390000000000000000,350.000000000000000000,103.000000000000000000,0.060000000000000000,3.330000000000000000,COMPLETED,2026-07-04T22:18:31.749Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-04,2026-10-02T14:44:58.547Z
22899404533812277248,BUY,USDT,INR,6.080000000000000000,630.000000000000000000,103.540000000000000000,0.060000000000000000,6.020000000000000000,COMPLETED,2026-06-14T09:19:15.226Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-06-14,2026-10-02T14:44:58.547Z
22898413188949139456,BUY,USDT,INR,6.050000000000000000,630.000000000000000000,103.990000000000000000,0.060000000000000000,5.990000000000000000,COMPLETED,2026-06-11T15:40:00.189Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-06-11,2026-10-02T14:44:58.547Z


In [0]:
(
    p2p_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_p2p_transactions"
    )
)

display(
    spark.table(
        f"{catalog}.{silver_schema}.binance_p2p_transactions"
    )
)

p2p_order_id,trade_type,asset,fiat,asset_amount,fiat_amount,unit_price,fee_amount,net_asset_amount,status,transaction_time,payment_method,exchange,_source_file,bronze_ingested_at,is_completed,transaction_date,silver_processed_at
22899404533812277248,BUY,USDT,INR,6.080000000000000000,630.000000000000000000,103.540000000000000000,0.060000000000000000,6.020000000000000000,COMPLETED,2026-06-14T09:19:15.226Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-06-14,2026-10-02T14:45:01.289Z
22898272405559799808,BUY,USDT,INR,5.070000000000000000,530.000000000000000000,104.450000000000000000,0.060000000000000000,5.010000000000000000,COMPLETED,2026-06-11T06:20:34.842Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-06-11,2026-10-02T14:45:01.289Z
22896216150807388160,BUY,USDT,INR,2.890000000000000000,300.000000000000000000,103.700000000000000000,0.060000000000000000,2.830000000000000000,COMPLETED,2026-06-05T14:09:45.470Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-06-05,2026-10-02T14:45:01.289Z
22909209175754317824,BUY,USDT,INR,3.420000000000000000,350.000000000000000000,102.200000000000000000,0.060000000000000000,3.360000000000000000,COMPLETED,2026-07-11T10:39:24.047Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-11,2026-10-02T14:45:01.289Z
22906848402065932288,BUY,USDT,INR,3.390000000000000000,350.000000000000000000,103.000000000000000000,0.060000000000000000,3.330000000000000000,COMPLETED,2026-07-04T22:18:31.749Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-04,2026-10-02T14:45:01.289Z
22918338709083369472,BUY,USDT,INR,3.880000000000000000,400.000000000000000000,102.940000000000000000,0.060000000000000000,3.820000000000000000,COMPLETED,2026-08-05T15:16:54.361Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-08-05,2026-10-02T14:45:01.289Z
22898377650986762240,BUY,USDT,INR,3.840000000000000000,400.000000000000000000,104.000000000000000000,0.060000000000000000,3.780000000000000000,CANCELLED,2026-06-11T13:18:47.316Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,false,2026-06-11,2026-10-02T14:45:01.289Z
22924149347226902528,BUY,USDT,INR,3.860000000000000000,400.000000000000000000,103.500000000000000000,0.060000000000000000,3.800000000000000000,COMPLETED,2026-08-21T16:06:18.475Z,UPI,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-08-21,2026-10-02T14:45:01.289Z
22895954482175311872,BUY,USDT,INR,4.800000000000000000,500.000000000000000000,104.000000000000000000,0.060000000000000000,4.740000000000000000,CANCELLED,2026-06-04T20:49:58.817Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,false,2026-06-04,2026-10-02T14:45:01.289Z
22907162745527037952,BUY,USDT,INR,3.500000000000000000,359.000000000000000000,102.300000000000000000,0.060000000000000000,3.440000000000000000,COMPLETED,2026-07-05T19:07:37.023Z,UPIQRCode,Binance,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,true,2026-07-05,2026-10-02T14:45:01.289Z


####Binance account

In [0]:
account_bronze_df  = spark.table(
    f"{catalog}.{bronze_schema}.binance_account_snapshots"
)

print(
    "Bronze account rows",account_bronze_df.count()
)

account_bronze_df.printSchema()

display(account_bronze_df)

Bronze account rows 2
root
 |-- makerCommission: long (nullable = true)
 |-- takerCommission: long (nullable = true)
 |-- buyerCommission: long (nullable = true)
 |-- sellerCommission: long (nullable = true)
 |-- commissionRates: struct (nullable = true)
 |    |-- maker: string (nullable = true)
 |    |-- taker: string (nullable = true)
 |    |-- buyer: string (nullable = true)
 |    |-- seller: string (nullable = true)
 |-- canTrade: boolean (nullable = true)
 |-- canWithdraw: boolean (nullable = true)
 |-- canDeposit: boolean (nullable = true)
 |-- brokered: boolean (nullable = true)
 |-- requireSelfTradePrevention: boolean (nullable = true)
 |-- preventSor: boolean (nullable = true)
 |-- updateTime: long (nullable = true)
 |-- accountType: string (nullable = true)
 |-- balances: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- asset: string (nullable = true)
 |    |    |-- free: string (nullable = true)
 |    |    |-- locked: string (nullable = 

makerCommission takerCommission buyerCommission sellerCommission commissionRates canTrade canWithdraw canDeposit brokered requireSelfTradePrevention preventSor updateTime accountType balances permissions uid source_file_name _source_file exchange source_type ingested_at 10 10 0 0 List(0.00100000, 0.00100000, 0.00000000, 0.00000000) true true true false false false 1788114758003 SPOT List(List(BTC, 0.00028971, 0.00000000), List(LTC, 0.00000000, 0.00000000), List(ETH, 0.00699300, 0.00000000), List(NEO, 0.00000000, 0.00000000), List(BNB, 0.00000000, 0.00000000), List(QTUM, 0.00000000, 0.00000000), List(EOS, 0.00000000, 0.00000000), List(SNT, 0.00000000, 0.00000000), List(BNT, 0.00000000, 0.00000000), List(GAS, 0.00000000, 0.00000000), List(BCC, 0.00000000, 0.00000000), List(USDT, 0.43099500, 0.00000000), List(HSR, 0.00000000, 0.00000000), List(OAX, 0.00000000, 0.00000000), List(DNT, 0.00000000, 0.00000000), List(MCO, 0.00000000, 0.00000000), List(ICN, 0.00000000, 0.00000000), List(ZRX, 0.00000000, 0.00000000), List(OMG, 0.00000000, 0.00000000), List(WTC, 0.00000000, 0.00000000), List(YOYO, 0.00000000, 0.00000000), List(LRC, 0.00000000, 0.00000000), List(TRX, 0.00000000, 0.00000000), List(SNGLS, 0.00000000, 0.00000000), List(STRAT, 0.00000000, 0.00000000), List(BQX, 0.00000000, 0.00000000), List(FUN, 0.00000000, 0.00000000), List(KNC, 0.00000000, 0.00000000), List(CDT, 0.00000000, 0.00000000), List(XVG, 0.00000000, 0.00000000), List(IOTA, 0.00000000, 0.00000000), List(SNM, 0.00000000, 0.00000000), List(LINK, 0.75924000, 0.00000000), List(CVC, 0.00000000, 0.00000000), List(TNT, 0.00000000, 0.00000000), List(REP, 0.00000000, 0.00000000), List(MDA, 0.00000000, 0.00000000), List(MTL, 0.00000000, 0.00000000), List(SALT, 0.00000000, 0.00000000), List(NULS, 0.00000000, 0.00000000), List(SUB, 0.00000000, 0.00000000), List(STX, 0.00000000, 0.00000000), List(MTH, 0.00000000, 0.00000000), List(ADX, 0.00000000, 0.00000000), List(ETC, 0.00000000, 0.00000000), List(ENG, 0.00000000, 0.00000000), List(ZEC, 0.00000000, 0.00000000), List(AST, 0.00000000, 0.00000000), List(GNT, 0.00000000, 0.00000000), List(DGD, 0.00000000, 0.00000000), List(BAT, 0.00000000, 0.00000000), List(DASH, 0.00000000, 0.00000000), List(POWR, 0.00000000, 0.00000000), List(BTG, 0.00000000, 0.00000000), List(REQ, 0.00000000, 0.00000000), List(XMR, 0.00000000, 0.00000000), List(EVX, 0.00000000, 0.00000000), List(VIB, 0.00000000, 0.00000000), List(ENJ, 0.00000000, 0.00000000), List(VEN, 0.00000000, 0.00000000), List(ARK, 0.00000000, 0.00000000), List(XRP, 0.00000000, 0.00000000), List(MOD, 0.00000000, 0.00000000), List(STORJ, 0.00000000, 0.00000000), List(KMD, 0.00000000, 0.00000000), List(RCN, 0.00000000, 0.00000000), List(EDO, 0.00000000, 0.00000000), List(DATA, 0.00000000, 0.00000000), List(DLT, 0.00000000, 0.00000000), List(MANA, 0.00000000, 0.00000000), List(PPT, 0.00000000, 0.00000000), List(RDN, 0.00000000, 0.00000000), List(GXS, 0.00000000, 0.00000000), List(AMB, 0.00000000, 0.00000000), List(ARN, 0.00000000, 0.00000000), List(BCPT, 0.00000000, 0.00000000), List(CND, 0.00000000, 0.00000000), List(GVT, 0.00000000, 0.00000000), List(POE, 0.00000000, 0.00000000), List(BTS, 0.00000000, 0.00000000), List(FUEL, 0.00000000, 0.00000000), List(XZC, 0.00000000, 0.00000000), List(QSP, 0.00000000, 0.00000000), List(LSK, 0.00000000, 0.00000000), List(BCD, 0.00000000, 0.00000000), List(TNB, 0.00000000, 0.00000000), List(ADA, 0.00000000, 0.00000000), List(LEND, 0.00000000, 0.00000000), List(XLM, 0.00000000, 0.00000000), List(CMT, 0.00000000, 0.00000000), List(WAVES, 0.00000000, 0.00000000), List(WABI, 0.00000000, 0.00000000), List(GTO, 0.00000000, 0.00000000), List(ICX, 0.00000000, 0.00000000), List(OST, 0.00000000, 0.00000000), List(ELF, 0.00000000, 0.00000000), List(AION, 0.00000000, 0.00000000), List(WINGS, 0.00000000, 0.00000000), List(BRD, 0.00000000, 0.00000000), List(NEBL, 0.00000000, 0.00000000), List(NAV, 0.00000000, 0.00000000), List(VIBE, 0.00000000, 0.00000000), List(LUN, 0

In [0]:
account_snapshots_silver_df = (
    account_bronze_df
    .select(
        col("source_file_name").alias("snapshot_id"),
        col("uid").cast("string").alias("account_id"),
        col("accountType").alias("account_type"),

        col("canTrade").alias("can_trade"),
        col("canWithdraw").alias("can_withdraw"),
        col("canDeposit").alias("can_deposit"),

        col("commissionRates.maker")
            .cast("decimal(18,8)")
            .alias("maker_commission_rate"),

        col("commissionRates.taker")
            .cast("decimal(18,8)")
            .alias("taker_commission_rate"),

        col("commissionRates.buyer")
            .cast("decimal(18,8)")
            .alias("buyer_commission_rate"),

        col("commissionRates.seller")
            .cast("decimal(18,8)")
            .alias("seller_commission_rate"),

        when(
            col("updateTime") > 0,
            timestamp_millis(
                col("updateTime")
            )
        ).alias("account_updated_at"),

        col("permissions"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print(
    "Clean account snapshot rows:",
    account_snapshots_silver_df.count()
)

display(account_snapshots_silver_df)

Clean account snapshot rows: 2


snapshot_id,account_id,account_type,can_trade,can_withdraw,can_deposit,maker_commission_rate,taker_commission_rate,buyer_commission_rate,seller_commission_rate,account_updated_at,permissions,exchange,_source_file,bronze_ingested_at,silver_processed_at
account_20261002T141739_738195Z_b4941cd6.json,1225138972,SPOT,true,true,true,0.00100000,0.00100000,0E-8,0E-8,2026-08-30T18:32:38.003Z,List(TRD_GRP_049),Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,2026-10-02T14:45:06.985Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,SPOT,true,true,true,0.00100000,0.00100000,0E-8,0E-8,2026-08-30T18:32:38.003Z,List(TRD_GRP_049),Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,2026-10-02T14:45:06.985Z


In [0]:
rows_before = account_snapshots_silver_df.count()

account_snapshots_final_df = (
    account_snapshots_silver_df
    .dropDuplicates([
        "snapshot_id"
    ])
)

rows_after = account_snapshots_final_df.count()

print("Rows before deduplication:", rows_before)
print("Rows after deduplication:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

Rows before deduplication: 2
Rows after deduplication: 2
Duplicate rows removed: 0


In [0]:
(
    account_snapshots_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_account_snapshots"
    )
)
print(
    "Saved account snapshots:",
    spark.table(
        f"{catalog}.{silver_schema}.binance_account_snapshots"
    ).count()
)

Saved account snapshots: 2


####Binance balances

In [0]:
account_balances_silver_df = (
    account_bronze_df
    .select(
        col("source_file_name").alias("snapshot_id"),
        col("uid").cast("string").alias("account_id"),
        explode(col("balances")).alias("balance"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .select(
        "snapshot_id",
        "account_id",

        upper(col("balance.asset")).alias("asset"),

        col("balance.free")
            .cast("decimal(38,18)")
            .alias("free_balance"),

        col("balance.locked")
            .cast("decimal(38,18)")
            .alias("locked_balance"),

        "exchange",
        "_source_file",
        "bronze_ingested_at"
    )
    .withColumn(
        "total_balance",
        col("free_balance") + col("locked_balance")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)

print(
    "Account balance rows:",
    account_balances_silver_df.count()
)

display(
    account_balances_silver_df.orderBy(
        col("total_balance").desc()
    )
)

Account balance rows: 1697


snapshot_id,account_id,asset,free_balance,locked_balance,exchange,_source_file,bronze_ingested_at,total_balance,silver_processed_at
account_20260925T182104_128404Z_4e8bc291.json,1225138972,ONDO,18.281700000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,18.28170000000000000,2026-10-02T14:45:13.611Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,ONDO,18.281700000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,18.28170000000000000,2026-10-02T14:45:13.611Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,SUI,8.091900000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,8.09190000000000000,2026-10-02T14:45:13.611Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,SUI,8.091900000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,8.09190000000000000,2026-10-02T14:45:13.611Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,LINK,0.759240000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,0.75924000000000000,2026-10-02T14:45:13.611Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,LINK,0.759240000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,0.75924000000000000,2026-10-02T14:45:13.611Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,USDT,0.430995000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,0.43099500000000000,2026-10-02T14:45:13.611Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,USDT,0.430995000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,0.43099500000000000,2026-10-02T14:45:13.611Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,SOL,0.114885000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,0.11488500000000000,2026-10-02T14:45:13.611Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,SOL,0.114885000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,0.11488500000000000,2026-10-02T14:45:13.611Z


In [0]:
active_balances_df = (
    account_balances_silver_df
    .filter(col("total_balance") > 0)
)

total_rows = account_balances_silver_df.count()
active_rows = active_balances_df.count()
zero_rows = total_rows - active_rows

print("Total balance rows:", total_rows)
print("Assets with funds:", active_rows)
print("Assets with zero balance:", zero_rows)

display(
    active_balances_df.orderBy(
        col("total_balance").desc()
    )
)

Total balance rows: 1697
Assets with funds: 14
Assets with zero balance: 1683


snapshot_id,account_id,asset,free_balance,locked_balance,exchange,_source_file,bronze_ingested_at,total_balance,silver_processed_at
account_20260925T182104_128404Z_4e8bc291.json,1225138972,ONDO,18.281700000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,18.28170000000000000,2026-10-02T14:45:15.784Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,ONDO,18.281700000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,18.28170000000000000,2026-10-02T14:45:15.784Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,SUI,8.091900000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,8.09190000000000000,2026-10-02T14:45:15.784Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,SUI,8.091900000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,8.09190000000000000,2026-10-02T14:45:15.784Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,LINK,0.759240000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,0.75924000000000000,2026-10-02T14:45:15.784Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,LINK,0.759240000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,0.75924000000000000,2026-10-02T14:45:15.784Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,USDT,0.430995000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,0.43099500000000000,2026-10-02T14:45:15.784Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,USDT,0.430995000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,0.43099500000000000,2026-10-02T14:45:15.784Z
account_20260925T182104_128404Z_4e8bc291.json,1225138972,SOL,0.114885000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:20.675Z,0.11488500000000000,2026-10-02T14:45:15.784Z
account_20261002T141739_738195Z_b4941cd6.json,1225138972,SOL,0.114885000000000000,0E-18,Binance,s3://l1l2bucket/data/raw/binance/account/account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:20.675Z,0.11488500000000000,2026-10-02T14:45:15.784Z


In [0]:
account_balances_final_df = (
    account_balances_silver_df
    .dropDuplicates(
        ["snapshot_id", "account_id", "asset"]
    )
)

(
    account_balances_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_account_balances"
    )
)

saved_balance_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_account_balances"
).count()

print("Saved account balance rows:", saved_balance_rows)

Saved account balance rows: 1697


####Binance deposits

In [0]:
deposits_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_deposits"
)

print(
    "Deposits rows: ",
    deposits_bronze_df.count()
)

deposits_bronze_df.printSchema()
display(deposits_bronze_df)

Deposits rows:  0
root
 |-- id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- coin: string (nullable = true)
 |-- network: string (nullable = true)
 |-- status: long (nullable = true)
 |-- address: string (nullable = true)
 |-- addressTag: string (nullable = true)
 |-- txId: string (nullable = true)
 |-- sourceAddress: string (nullable = true)
 |-- insertTime: long (nullable = true)
 |-- completeTime: long (nullable = true)
 |-- transferType: long (nullable = true)
 |-- confirmTimes: string (nullable = true)
 |-- unlockConfirm: long (nullable = true)
 |-- walletType: long (nullable = true)
 |-- travelRuleStatus: long (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



id,amount,coin,network,status,address,addressTag,txId,sourceAddress,insertTime,completeTime,transferType,confirmTimes,unlockConfirm,walletType,travelRuleStatus,source_file_name,_source_file,exchange,source_type,ingested_at


In [0]:
deposits_silver_df = (
    deposits_bronze_df
    .select(
        col("id").alias("deposit_id"),
        col("txId").alias("transaction_hash"),
        upper(col("coin")).alias("asset"),
        col("amount")
            .cast("decimal(38,18)")
            .alias("amount"),
        upper(col("network")).alias("network"),
        col("status")
            .cast("integer")
            .alias("status_code"),

        col("address").alias("deposit_address"),
        col("addressTag").alias("address_tag"),
        col("sourceAddress").alias("source_address"),

        when(
            col("insertTime") > 0,
            timestamp_millis(
                col("insertTime")
            )
        ).alias("deposit_time"),

        when(
            col("completeTime") > 0,
            timestamp_millis(
                col("completeTime")
            )
        ).alias("completed_at"),

        col("transferType")
            .cast("integer")
            .alias("transfer_type"),

        col("confirmTimes").alias("confirmations"),

        col("unlockConfirm")
            .cast("integer")
            .alias("unlock_confirmations"),

        col("walletType")
            .cast("integer")
            .alias("wallet_type"),

        col("travelRuleStatus")
            .cast("integer")
            .alias("travel_rule_status"),

        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print("Clean deposit rows:", deposits_silver_df.count())

deposits_silver_df.printSchema()

Clean deposit rows: 0
root
 |-- deposit_id: string (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- network: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- deposit_address: string (nullable = true)
 |-- address_tag: string (nullable = true)
 |-- source_address: string (nullable = true)
 |-- deposit_time: timestamp (nullable = true)
 |-- completed_at: timestamp (nullable = true)
 |-- transfer_type: integer (nullable = true)
 |-- confirmations: string (nullable = true)
 |-- unlock_confirmations: integer (nullable = true)
 |-- wallet_type: integer (nullable = true)
 |-- travel_rule_status: integer (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
deposits_final_df = (
    deposits_silver_df
    .dropDuplicates(["deposit_id"])
)

print(
    "Duplicate deposit rows removed:",
    deposits_silver_df.count() - deposits_final_df.count()
)

(
    deposits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_deposits"
    )
)

saved_deposit_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_deposits"
).count()

print("Saved deposit rows:", saved_deposit_rows)

Duplicate deposit rows removed: 0
Saved deposit rows: 0


####Binance withdrawals

In [0]:
withdrawals_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_withdrawals"
)

print(
    "Bronze withdrawal rows:",
    withdrawals_bronze_df.count()
)

withdrawals_bronze_df.printSchema()

display(withdrawals_bronze_df)

Bronze withdrawal rows: 1
root
 |-- id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- transactionFee: string (nullable = true)
 |-- coin: string (nullable = true)
 |-- status: long (nullable = true)
 |-- address: string (nullable = true)
 |-- txId: string (nullable = true)
 |-- applyTime: string (nullable = true)
 |-- network: string (nullable = true)
 |-- transferType: long (nullable = true)
 |-- withdrawOrderId: string (nullable = true)
 |-- info: string (nullable = true)
 |-- confirmNo: long (nullable = true)
 |-- walletType: long (nullable = true)
 |-- txKey: string (nullable = true)
 |-- completeTime: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



id,amount,transactionFee,coin,status,address,txId,applyTime,network,transferType,withdrawOrderId,info,confirmNo,walletType,txKey,completeTime,source_file_name,_source_file,exchange,source_type,ingested_at
3f2234d4647746d4ac478d1e8d792855,0.10665482,0.001,SOL,6,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,kGWB1m8K4ZvZYRGncfZMv3A1okHhKMrwacGfUn9FuEnLButrU92rZFj2xqwV48ftYwJQV5qAYwUZrCeecaaSvfU,2026-03-13 06:39:17,SOL,0,null,"blockhash:FsvRc4r4qqAZApYTGEKmy8QhVGH7wt2fsXUuyx3cbgBa,nonceAccount:3kQUuRN7wu6fhfpxpKjszWKXpeE1kav58aoMsXZ9sciL",2,0,,2026-03-13 06:41:06,withdrawals_20260925T182120_839679Z_eb18a75d.json,s3://l1l2bucket/data/raw/binance/withdrawals/withdrawals_20260925T182120_839679Z_eb18a75d.json,Binance,Withdrawal,2026-10-02T14:22:43.784Z


In [0]:
withdrawals_silver_df = (
    withdrawals_bronze_df
    .select(
        col("id").alias("withdrawal_id"),
        col("withdrawOrderId").alias("withdrawal_order_id"),
        col("txId").alias("transaction_hash"),

        upper(col("coin")).alias("asset"),

        col("amount")
            .cast("decimal(38,18)")
            .alias("amount"),

        col("transactionFee")
            .cast("decimal(38,18)")
            .alias("transaction_fee"),

        upper(col("network")).alias("network"),

        col("status")
            .cast("integer")
            .alias("status_code"),

        col("address").alias("withdrawal_address"),

        to_timestamp(
            col("applyTime")
        ).alias("requested_at"),

        to_timestamp(
            col("completeTime")
        ).alias("completed_at"),

        col("confirmNo")
            .cast("integer")
            .alias("confirmation_count"),

        col("transferType")
            .cast("integer")
            .alias("transfer_type"),

        col("walletType")
            .cast("integer")
            .alias("wallet_type"),

        col("txKey").alias("transaction_key"),
        col("info").alias("status_information"),

        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print(
    "Clean withdrawal rows:",
    withdrawals_silver_df.count()
)

display(withdrawals_silver_df)

Clean withdrawal rows: 1


withdrawal_id,withdrawal_order_id,transaction_hash,asset,amount,transaction_fee,network,status_code,withdrawal_address,requested_at,completed_at,confirmation_count,transfer_type,wallet_type,transaction_key,status_information,exchange,_source_file,bronze_ingested_at,silver_processed_at
3f2234d4647746d4ac478d1e8d792855,null,kGWB1m8K4ZvZYRGncfZMv3A1okHhKMrwacGfUn9FuEnLButrU92rZFj2xqwV48ftYwJQV5qAYwUZrCeecaaSvfU,SOL,0.106654820000000000,0.001000000000000000,SOL,6,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,2026-03-13T06:39:17.000Z,2026-03-13T06:41:06.000Z,2,0,0,,"blockhash:FsvRc4r4qqAZApYTGEKmy8QhVGH7wt2fsXUuyx3cbgBa,nonceAccount:3kQUuRN7wu6fhfpxpKjszWKXpeE1kav58aoMsXZ9sciL",Binance,s3://l1l2bucket/data/raw/binance/withdrawals/withdrawals_20260925T182120_839679Z_eb18a75d.json,2026-10-02T14:22:43.784Z,2026-10-02T14:45:29.419Z


In [0]:
withdrawal_rows_before = withdrawals_silver_df.count()

withdrawals_final_df = (
    withdrawals_silver_df
    .dropDuplicates(["withdrawal_id"])
)

withdrawal_rows_after = withdrawals_final_df.count()

print("Rows before deduplication:", withdrawal_rows_before)
print("Rows after deduplication:", withdrawal_rows_after)
print(
    "Duplicate withdrawal rows removed:",
    withdrawal_rows_before - withdrawal_rows_after
)

Rows before deduplication: 1
Rows after deduplication: 1
Duplicate withdrawal rows removed: 0


In [0]:
(
    withdrawals_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_withdrawals"
    )
)

saved_withdrawal_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_withdrawals"
).count()

print("Saved withdrawal rows:", saved_withdrawal_rows)

Saved withdrawal rows: 1


####Binance internal transfer

In [0]:
transfers_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_transfers"
)

print(
    "Bronze transfer rows:",
    transfers_bronze_df.count()
)

transfers_bronze_df.printSchema()

display(transfers_bronze_df)

Bronze transfer rows: 0
root
 |-- tranId: long (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- transferType: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



tranId,asset,amount,type,status,timestamp,transferType,source_file_name,_source_file,exchange,source_type,ingested_at


In [0]:
transfers_silver_df = (
    transfers_bronze_df
    .select(
        col("tranId")
            .cast("string")
            .alias("transfer_id"),

        upper(col("asset")).alias("asset"),

        col("amount")
            .cast("decimal(38,18)")
            .alias("amount"),

        upper(col("type")).alias("transfer_type"),

        upper(col("transferType"))
            .alias("requested_transfer_type"),

        upper(col("status")).alias("status"),

        when(
            col("timestamp") > 0,
            timestamp_millis(
                col("timestamp")
            )
        ).alias("transfer_time"),

        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print(
    "Clean transfer rows:",
    transfers_silver_df.count()
)

transfers_silver_df.printSchema()

Clean transfer rows: 0
root
 |-- transfer_id: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- transfer_type: string (nullable = true)
 |-- requested_transfer_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- transfer_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
transfers_final_df = (
    transfers_silver_df
    .dropDuplicates(["transfer_id"])
)

print(
    "Duplicate transfer rows removed:",
    transfers_silver_df.count() - transfers_final_df.count()
)

(
    transfers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_transfers"
    )
)

saved_transfer_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_transfers"
).count()

print("Saved transfer rows:", saved_transfer_rows)

Duplicate transfer rows removed: 0
Saved transfer rows: 0


####Binance price history

In [0]:
price_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_price_history"
)

print("Bronze price rows:", price_bronze_df.count())

print("BRONZE SCHEMA")
price_bronze_df.printSchema()

Bronze price rows: 2952
BRONZE SCHEMA
root
 |-- symbol: string (nullable = true)
 |-- open_time: string (nullable = true)
 |-- open_price: string (nullable = true)
 |-- high_price: string (nullable = true)
 |-- low_price: string (nullable = true)
 |-- close_price: string (nullable = true)
 |-- volume: string (nullable = true)
 |-- close_time: string (nullable = true)
 |-- quote_asset_volume: string (nullable = true)
 |-- number_of_trades: string (nullable = true)
 |-- taker_buy_base_volume: string (nullable = true)
 |-- taker_buy_quote_volume: string (nullable = true)
 |-- unused_field: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- source_file_saved_at: timestamp (nullable = true)
 |-- source_file_modified_at: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
price_silver_df = (
    price_bronze_df
    .select(
        upper(
            col("symbol")
        ).alias("symbol"),

        regexp_replace(
            upper(col("symbol")),
            "USDT$",
            ""
        ).alias("asset"),

        timestamp_millis(
            col("open_time").cast("long")
        ).alias("open_time"),

        timestamp_millis(
            col("close_time").cast("long")
        ).alias("close_time"),

        col("open_price")
            .cast("decimal(38,18)")
            .alias("open_price"),

        col("high_price")
            .cast("decimal(38,18)")
            .alias("high_price"),

        col("low_price")
            .cast("decimal(38,18)")
            .alias("low_price"),

        col("close_price")
            .cast("decimal(38,18)")
            .alias("close_price"),

        col("volume")
            .cast("decimal(38,18)")
            .alias("volume"),

        col("quote_asset_volume")
            .cast("decimal(38,18)")
            .alias("quote_asset_volume"),

        col("number_of_trades")
            .cast("long")
            .alias("trade_count"),

        col("taker_buy_base_volume")
            .cast("decimal(38,18)")
            .alias("taker_buy_base_volume"),

        col("taker_buy_quote_volume")
            .cast("decimal(38,18)")
            .alias("taker_buy_quote_volume"),

        col("exchange"),
        col("_source_file"),
        col("source_file_modified_at"),
        col("source_file_saved_at"),
        col("ingested_at")
            .alias("bronze_ingested_at"),

        current_timestamp()
            .alias("silver_processed_at")
    )
)

print(
    "Clean price rows:",
    price_silver_df.count()
)

display(
    price_silver_df.orderBy(
        col("open_time").desc()
    )
)

Clean price rows: 2952


symbol,asset,open_time,close_time,open_price,high_price,low_price,close_price,volume,quote_asset_volume,trade_count,taker_buy_base_volume,taker_buy_quote_volume,exchange,_source_file,source_file_modified_at,source_file_saved_at,bronze_ingested_at,silver_processed_at
LINKUSDT,LINK,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,14.385000000000000000,14.652000000000000000,14.164000000000000000,14.330000000000000000,2096272.630000000000000000,30191700.410280000000000000,233705,1035042.530000000000000000,14911135.702460000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/LINKUSDT/price_history_20261002T141805_809086Z_411b4afe.json,2026-10-02T14:18:05.000Z,2026-10-02T14:18:05.809Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
SOLUSDT,SOL,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,118.410000000000000000,123.760000000000000000,118.280000000000000000,121.950000000000000000,2060781.109000000000000000,250996758.791600000000000000,677185,1105927.059000000000000000,134639187.195690000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/SOLUSDT/price_history_20261002T141804_758297Z_2df7d944.json,2026-10-02T14:18:04.000Z,2026-10-02T14:18:04.758Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
ETHUSDT,ETH,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,2706.430000000000000000,2777.330000000000000000,2696.560000000000000000,2738.720000000000000000,226154.564500000000000000,619661901.583315000000000000,2610080,116873.930800000000000000,320135257.194681000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/ETHUSDT/price_history_20261002T141804_360765Z_207c580a.json,2026-10-02T14:18:03.000Z,2026-10-02T14:18:04.360Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
SUIUSDT,SUI,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,1.177200000000000000,1.218600000000000000,1.163000000000000000,1.186700000000000000,54223248.200000000000000000,64482395.634370000000000000,462689,28466524.800000000000000000,33868676.946220000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/SUIUSDT/price_history_20261002T141805_284183Z_b09085f0.json,2026-10-02T14:18:04.000Z,2026-10-02T14:18:05.284Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
BTCUSDT,BTC,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,84880.050000000000000000,87220.000000000000000000,84520.000000000000000000,86447.530000000000000000,15148.284610000000000000,1305282019.229985000000000000,2315898,7011.464710000000000000,604129254.759916400000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.000Z,2026-10-02T14:18:03.901Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
ONDOUSDT,ONDO,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,0.497200000000000000,0.516200000000000000,0.489600000000000000,0.506500000000000000,34221727.900000000000000000,17316882.661230000000000000,83798,16609635.500000000000000000,8403755.947520000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/ONDOUSDT/price_history_20261002T141806_205850Z_0b2ff977.json,2026-10-02T14:18:05.000Z,2026-10-02T14:18:06.205Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
LINKUSDT,LINK,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,14.366000000000000000,14.560000000000000000,14.142000000000000000,14.386000000000000000,2480980.420000000000000000,35591436.303330000000000000,339056,1246803.460000000000000000,17890444.220600000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/LINKUSDT/price_history_20261002T141805_809086Z_411b4afe.json,2026-10-02T14:18:05.000Z,2026-10-02T14:18:05.809Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:43.442Z
SUIUSDT,SUI,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,1.164300000000000000,1.198600000000000000,1.127100000000000000,1.177300000000000000,74933608.900000000000000000,87280951.672700000000000000,694214,37146889.500000000000000000,43304417.222490000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/SUIUSDT/pr

In [0]:
price_enriched_df = (
    price_silver_df
    .withColumn(
        "is_closed",
        col("close_time") <= col("source_file_saved_at")
    )
    .withColumn(
        "price_change",
        (
            col("close_price") - col("open_price")
        ).cast("decimal(38,18)")
    )
    .withColumn(
        "price_change_pct",
        when(
            col("open_price") != 0,
            (
                (
                    col("close_price") - col("open_price")
                ) / col("open_price")
            ) * 100
        )
        .otherwise(None)
        .cast("decimal(38,18)")
    )
)


price_rows_before = price_enriched_df.count()


price_final_df = (
    price_enriched_df
    .dropDuplicates([
        "symbol",
        "open_time"
    ])
)


price_rows_after = price_final_df.count()


print("Rows before deduplication:", price_rows_before)
print("Rows after deduplication:", price_rows_after)
print(
    "Duplicate candles removed:",
    price_rows_before - price_rows_after
)


price_final_df.groupBy(
    "is_closed"
).count().show()

Rows before deduplication: 2952
Rows after deduplication: 2952
Duplicate candles removed: 0
+---------+-----+
|is_closed|count|
+---------+-----+
|     true| 2946|
|    false|    6|
+---------+-----+



In [0]:
(
    price_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_price_history"
    )
)

print(
    "Saved price history rows:",
    spark.table(
        f"{catalog}.{silver_schema}.binance_price_history"
    ).count()
)

Saved price history rows: 2952


####Binance funding account snapshots

In [0]:
funding_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_funding_account_snapshots"
)

funding_snapshots_silver_df = (
    funding_bronze_df
    .select(
        col("source_file_name").alias(
            "snapshot_id"
        ),

        upper(col("walletType")).alias(
            "wallet_type"
        ),

        to_timestamp(
            col("capturedAt")
        ).alias("snapshot_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at").alias(
            "bronze_ingested_at"
        ),

        current_timestamp().alias(
            "silver_processed_at"
        )
    )
    .dropDuplicates([
        "snapshot_id"
    ])
)

print(
    "Funding snapshot rows:",
    funding_snapshots_silver_df.count()
)

display(
    funding_snapshots_silver_df.orderBy(
        col("snapshot_time").desc()
    )
)

Funding snapshot rows: 2


snapshot_id,wallet_type,snapshot_time,exchange,_source_file,bronze_ingested_at,silver_processed_at
funding_account_20261002T141740_508437Z_7aee3462.json,FUNDING,2026-10-02T14:17:40.159Z,Binance,s3://l1l2bucket/data/raw/binance/funding_account/funding_account_20261002T141740_508437Z_7aee3462.json,2026-10-02T14:23:33.808Z,2026-10-02T14:45:52.119Z
funding_account_20260925T182104_949722Z_6b29dd5e.json,FUNDING,2026-09-25T18:21:04.576Z,Binance,s3://l1l2bucket/data/raw/binance/funding_account/funding_account_20260925T182104_949722Z_6b29dd5e.json,2026-10-02T14:23:33.808Z,2026-10-02T14:45:52.119Z


In [0]:
(
    funding_snapshots_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_funding_account_snapshots"
    )
)

saved_funding_snapshots_df = spark.table(
    f"{catalog}.{silver_schema}.binance_funding_account_snapshots"
)

print(
    "Saved Funding snapshot rows:",
    saved_funding_snapshots_df.count()
)

Saved Funding snapshot rows: 2


#### Binance funding account balances

In [0]:
funding_balances_silver_df = (
    funding_bronze_df
    .select(
        col("source_file_name").alias(
            "snapshot_id"
        ),

        upper(col("walletType")).alias(
            "wallet_type"
        ),

        to_timestamp(
            col("capturedAt")
        ).alias("snapshot_time"),

        explode(col("balances")).alias(
            "balance"
        ),

        col("exchange"),
        col("_source_file"),

        col("ingested_at").alias(
            "bronze_ingested_at"
        )
    )
    .select(
        "snapshot_id",
        "wallet_type",
        "snapshot_time",

        upper(col("balance.asset")).alias(
            "asset"
        ),

        col("balance.free")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("balance.locked")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        col("balance.freeze")
        .cast("decimal(38,18)")
        .alias("frozen_balance"),

        col("balance.withdrawing")
        .cast("decimal(38,18)")
        .alias("withdrawing_balance"),

        col("exchange"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
    .withColumn(
        "total_balance",

        coalesce(
            col("free_balance"),
            lit(0)
        )
        + coalesce(
            col("locked_balance"),
            lit(0)
        )
        + coalesce(
            col("frozen_balance"),
            lit(0)
        )
        + coalesce(
            col("withdrawing_balance"),
            lit(0)
        )
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "snapshot_id",
        "asset"
    ])
)

print(
    "Funding balance rows:",
    funding_balances_silver_df.count()
)

funding_balances_silver_df.printSchema()

display(funding_balances_silver_df)

Funding balance rows: 0
root
 |-- snapshot_id: string (nullable = true)
 |-- wallet_type: string (nullable = true)
 |-- snapshot_time: timestamp (nullable = true)
 |-- asset: string (nullable = true)
 |-- free_balance: decimal(38,18) (nullable = true)
 |-- locked_balance: decimal(38,18) (nullable = true)
 |-- frozen_balance: decimal(38,18) (nullable = true)
 |-- withdrawing_balance: decimal(38,18) (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- total_balance: decimal(38,15) (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)



snapshot_id,wallet_type,snapshot_time,asset,free_balance,locked_balance,frozen_balance,withdrawing_balance,exchange,_source_file,bronze_ingested_at,total_balance,silver_processed_at


In [0]:
(
    funding_balances_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_funding_account_balances"
    )
)

saved_funding_balances_df = spark.table(
    f"{catalog}.{silver_schema}.binance_funding_account_balances"
)

print(
    "Saved Funding balance rows:",
    saved_funding_balances_df.count()
)

Saved Funding balance rows: 0


In [0]:
silver_tables = spark.sql(
    "SHOW TABLES IN crypto_lakehouse.silver"
)

display(silver_tables)

database,tableName,isTemporary
silver,binance_account_balances,false
silver,binance_account_snapshots,false
silver,binance_deposits,false
silver,binance_funding_account_balances,false
silver,binance_funding_account_snapshots,false
silver,binance_p2p_transactions,false
silver,binance_price_history,false
silver,binance_trades,false
silver,binance_transfers,false
silver,binance_withdrawals,false
